# arXiv paper scraper: 400 papers → seeds.jsonl

Scrapes arXiv API (default cs.CL / cs.LG, configurable) and stores paper metadata. No GPU needed, Internet ON.

In [ ]:
import os
ON_KAGGLE = os.path.exists('/kaggle/working')
ROOT = '/kaggle/working/arxiv_out' if ON_KAGGLE else './data/arxiv_400'
os.makedirs(ROOT, exist_ok=True)
print('root:', ROOT)
!pip install -q arxiv 2>&1 | tail -1


In [ ]:
# Cell 2 — scrape seeds via OAI-PMH bulk harvest
# (the search API 406s pagination AND throttles single queries;
# OAI with resumption tokens is the endpoint built for this).
import json, time, datetime
import requests
import xml.etree.ElementTree as ET

WANT_CATS = {"cs.CL", "cs.LG"}   # change to your topics
N_SEEDS = 400
UA = "PaperRec-Course-Project/1.0 (mailto:student@example.edu)"
OAI = "https://export.arxiv.org/oai2"
O = "http://www.openarchives.org/OAI/2.0/"
A = "http://arxiv.org/OAI/arXiv/"
SINCE = (datetime.date.today() - datetime.timedelta(days=90)).isoformat()

def author_name(a):
    key = (a.findtext(f"{{{A}}}keyname") or "").strip()
    fore = (a.findtext(f"{{{A}}}forenames") or "").strip()
    return f"{fore} {key}".strip()

def record_to_paper(rec):
    hdr = rec.find(f"{{{O}}}header")
    if hdr is not None and hdr.get("status") == "deleted":
        return None
    m = rec.find(f".//{{{A}}}arXiv")
    if m is None:
        return None
    pid = (m.findtext(f"{{{A}}}id") or "").strip()
    cats = (m.findtext(f"{{{A}}}categories") or "").split()
    if not (set(cats) & WANT_CATS):
        return None
    created = (m.findtext(f"{{{A}}}created") or "")[:4]
    au = m.find(f"{{{A}}}authors")
    authors = [author_name(a) for a in (au.findall(f"{{{A}}}author") if au is not None else [])]
    return {
        "source": "arxiv",
        "external_id": pid,
        "title": " ".join(((m.findtext(f"{{{A}}}title") or "").split())),
        "abstract": " ".join(((m.findtext(f"{{{A}}}abstract") or "").split())),
        "authors": [a for a in authors if a],
        "year": int(created) if created.isdigit() else None,
        "pdf_url": f"https://arxiv.org/pdf/{pid}" if pid else "",
        "categories": cats,
    }

seeds, seen, pages = [], set(), 0
params = {"verb": "ListRecords", "set": "cs", "metadataPrefix": "arXiv", "from": SINCE}
while len(seeds) < N_SEEDS:
    r = requests.get(OAI, params=params, headers={"User-Agent": UA}, timeout=180)
    r.raise_for_status()
    root = ET.fromstring(r.content)
    for rec in root.findall(f".//{{{O}}}record"):
        p = record_to_paper(rec)
        if p and p["external_id"] not in seen and p["abstract"]:
            seen.add(p["external_id"])
            seeds.append(p)
            if len(seeds) >= N_SEEDS:
                break
    tok = root.find(f".//{{{O}}}resumptionToken")
    token = (tok.text or "").strip() if tok is not None else ""
    pages += 1
    print(f"page {pages}: seeds={len(seeds)}", flush=True)
    if not token or len(seeds) >= N_SEEDS:
        break
    params = {"verb": "ListRecords", "resumptionToken": token}
    time.sleep(5.0)
print("seeds:", len(seeds))
assert len(seeds) >= N_SEEDS, f"only got {len(seeds)}"
with open(f"{ROOT}/seeds.jsonl", "w") as _f:
    _f.write("\n".join(json.dumps(s) for s in seeds) + "\n")
print("years:", sorted({s["year"] for s in seeds if s["year"]})[:3], "...")


In [ ]:
# Store + verify. On Kaggle: File → Save Version → attach output.
!ls -la "{ROOT}"
import json as _j
seeds = [_j.loads(l) for l in open(f"{ROOT}/seeds.jsonl")]
print(f"stored {len(seeds)} papers → {ROOT}/seeds.jsonl")
print("sample:", seeds[0]["title"][:100])


In [ ]:
!which aria2c >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -qq aria2) 2>&1 | tail -1
!pip install -q pypdf 2>&1 | tail -1
import json as _j, glob, os as _os
_os.makedirs(f"{ROOT}/pdfs", exist_ok=True)
seeds = [_j.loads(l) for l in open(f"{ROOT}/seeds.jsonl")]
urls = [s["pdf_url"] for s in seeds if s.get("pdf_url")]
open(f"{ROOT}/urls.txt", "w").write("\n".join(urls) + "\n")
print(f"downloading {len(urls)} pdfs with aria2...")
!cd "{ROOT}" && aria2c -i urls.txt -d pdfs -j16 -x8 --retry-wait=2 --max-tries=3 --timeout=60 -q 2>&1 | tail -2
from pypdf import PdfReader
out, n_ok = [], 0
for s in seeds:
    rec = dict(s)
    fname = (s.get("pdf_url") or "").split("/")[-1]
    pdf = f"{ROOT}/pdfs/" + fname
    if _os.path.exists(pdf):
        try:
            pages = [(pg.extract_text() or "") for pg in PdfReader(pdf).pages[:8]]
            rec["fulltext"] = " ".join("\n".join(pages).split())[:4000]
            n_ok += 1
        except Exception:
            rec["fulltext"] = ""
    else:
        rec["fulltext"] = ""
    out.append(rec)
with open(f"{ROOT}/papers.jsonl", "w") as _f:
    _f.write("\n".join(_j.dumps(r) for r in out) + "\n")
print(f"saved {len(out)} records with fulltext ok: {n_ok}")


In [ ]:
# Cell 5 - triplets for fine-tuning (anchor, positive, negative)
# Positives = cited references via Semantic Scholar; fallback = same-category pairs.
import json as _j, random, time
!pip install -q pandas requests 2>&1 | tail -1
import pandas as _pd
import requests as _rq
random.seed(42)
S2 = "https://api.semanticscholar.org/graph/v1"
UA2 = "PaperRec-Course-Project/1.0 (mailto:student@example.edu)"
MAX_REFS = 5

papers = [_j.loads(l) for l in open(f"{ROOT}/papers.jsonl")]
by_id = {x["external_id"]: x for x in papers}
print(f"corpus={len(by_id)}")

def _s2(path, params):
    try:
        r = _rq.get(f"{S2}{path}", params=params, headers={"User-Agent": UA2}, timeout=20)
        time.sleep(1.0)
        return r.json() if r.status_code == 200 else {}
    except Exception:
        return {}

edges, extra = [], {}
for i, s in enumerate(papers):
    aid = s["external_id"].split("v")[0]
    meta = _s2(f"/paper/arXiv:{aid}", {"fields": "paperId"})
    if meta.get("paperId"):
        items = _s2(f"/paper/{meta['paperId']}/references", {"fields": "paperId,title,abstract", "limit": 100}).get("data", [])
        for item in items[:MAX_REFS]:
            c = item.get("citedPaper", {})
            if c.get("paperId") and c.get("title"):
                edges.append((s["external_id"], c["paperId"]))
                if c["paperId"] not in by_id:
                    extra[c["paperId"]] = {"source": "semantic_scholar", "external_id": c["paperId"],
                        "title": c["title"], "abstract": c.get("abstract") or "", "authors": [],
                        "year": None, "pdf_url": "", "categories": s.get("categories", []), "fulltext": ""}
    if (i + 1) % 50 == 0:
        print(f"{i + 1}/{len(papers)} edges={len(edges)}", flush=True)
by_id.update(extra)
print(f"edges={len(edges)} corpus={len(by_id)}")

def txt(x):
    return f"{(x.get('title') or ' ').strip()} [SEP] {(x.get('abstract') or ' ').strip()}"

by_anchor = {}
for a, b in edges:
    if a in by_id and b in by_id and by_id[b].get("abstract"):
        by_anchor.setdefault(a, []).append(b)
if len(by_anchor) < 20:
    print("citation graph thin -> category fallback")
    catmap = {}
    for pid, x in by_id.items():
        for c in x.get("categories", [])[:2]:
            catmap.setdefault(c, []).append(pid)
    for pid, x in by_id.items():
        if not x.get("abstract"):
            continue
        for c in x.get("categories", [])[:1]:
            for other in catmap.get(c, [])[:3]:
                if other != pid and by_id[other].get("abstract"):
                    by_anchor.setdefault(pid, []).append(other)

ids = list(by_id)
rows = []
for a, positives in by_anchor.items():
    if not by_id[a].get("abstract"):
        continue
    for pos in positives[:3]:
        for _ in range(2):
            neg = random.choice(ids)
            if neg == a or neg in positives or not by_id[neg].get("abstract"):
                continue
            rows.append({"anchor_text": txt(by_id[a]), "positive_text": txt(by_id[pos]),
                         "negative_text": txt(by_id[neg])})
            break
df = _pd.DataFrame(rows).drop_duplicates()
df.to_csv(f"{ROOT}/triplets.csv", index=False)
print(f"triplets={len(df)} anchors={len(by_anchor)} -> {ROOT}/triplets.csv")
